# Training the UoM classifier

Maps noisy OCR'd Ukrainian unit-of-measure strings (`пакува rhh`, `ki`, `фл all.`) to canonical units.
The model is a softmax regression over character n-grams — small enough to ship as a JSON file and run in pure Python.

Pipeline: **dataset → augmentation → char-n-gram features → training → 5-fold CV → confidence threshold → artifact**.


In [ ]:
import sys; sys.path.insert(0, '../src')
import json, random
from pathlib import Path
from uom_classifier.train import (load_dataset, with_augmentation, vectorize,
                                  train_softmax, predict, pick_threshold, augment, SEED)

campaign, base, vocabulary, classes = load_dataset(Path('../data/uom_training_pairs.json'))
print(f'{len(campaign)} labeled OCR pairs, {len(base)} vocabulary pairs, {len(classes)} classes')


## 1. The data

`pairs` — real OCR forms with production-verified labels; `vocabulary` — exact spellings; `canons` — the label set.


In [ ]:
import collections
print('label distribution:', collections.Counter(l for _, l in campaign).most_common(8))
print('sample pairs:', campaign[:8])


## 2. OCR-style augmentation

Every training form is corrupted several times the way OCR corrupts text: Cyrillic↔Latin homoglyphs, stray dots, inserted spaces, random case.


In [ ]:
rng = random.Random(0)
print([augment('упаковка', rng) for _ in range(6)])


## 3. 5-fold cross-validation

The vocabulary always stays in the training half — the model has to prove itself on **campaign forms it never saw**.
The confidence threshold is chosen per fold for precision ≥ 0.98; the artifact ships the median.


In [ ]:
import numpy as np
rng = random.Random(SEED); rng.shuffle(campaign)
K = 5; folds = [campaign[i::K] for i in range(K)]
fold_thresholds, cv = [], [0, 0, 0]
for k in range(K):
    test = folds[k]
    train_pairs = with_augmentation([p for i, f in enumerate(folds) if i != k for p in f] + base,
                                    random.Random(SEED + k))
    fi = {}
    x, y = vectorize(train_pairs, fi, classes, grow=True)
    w, b = train_softmax(x, y, len(classes), epochs=800)
    forms = [f for f, _ in test]
    labels = np.array([classes.index(l) for _, l in test])
    pred, probs = predict(forms, w, b, fi)
    correct = pred == labels
    thr = pick_threshold(probs, correct)
    mask = probs >= thr
    fold_thresholds.append(thr)
    cv = [cv[0] + len(test), cv[1] + int(mask.sum()), cv[2] + int(correct[mask].sum())]
    print(f'fold {k}: thr={thr:.3f} coverage={mask.mean():.1%} precision={correct[mask].mean():.1%}')
print(f'CV: coverage={cv[1]/cv[0]:.1%} precision={cv[2]/cv[1]:.1%} | threshold={np.median(fold_thresholds):.3f}')


## 4. Final training + artifact

Final weights are trained on **all** data; the CV numbers above are what goes into `metrics`.


In [ ]:
from uom_classifier.train import train
metrics = train(Path('../data/uom_training_pairs.json'),
                Path('../src/uom_classifier/data/uom_classifier.json'))
metrics


## 5. Use it


In [ ]:
from uom_classifier import UomClassifier
clf = UomClassifier('../src/uom_classifier/data/uom_classifier.json')
for form in ['пакува rhh', 'фл all.', 'пляшк', 'шт.', 'шт/уп', 'дослідження', 'random garbage']:
    print(f'{form!r:20} -> {clf.classify(form)}')
